# Notebook 1 — How Practical GenAI Business Applications Work

### Building Practical GenAI Solutions for Business: *Build an AI Business Copilot* 🏔️

Welcome! Over two evenings you'll build a working **AI business copilot** for a fictional
outdoor retailer, **Trailhead Supply Co.** — an assistant that can answer real customer
questions like *"What's your return policy on final sale items?"* and *"Where is order #1007?"*

**This first notebook** sets the foundation. By the end you'll be able to:

1. Explain what a large language model (LLM) is — and, importantly, what it is **not**.
2. Make your first call to Claude and read its response.
3. Steer Claude's behavior with a **system prompt**.
4. See first-hand why *an LLM alone doesn't know your business data* — and understand how
   we'll fix that by **grounding** the model, **without ever training it**.

> 🐍 **No Python experience needed.** You'll run cells (▶️) and occasionally edit a value
> or a prompt where you see a ✏️ **Try it** marker. Read the text above each code cell —
> it explains what the code does and why.

## 1. Start with the business problem

Trailhead Supply Co. sells tents, backpacks, boots, and camp stoves online. Their small
support team is overwhelmed by the **same questions, over and over**:

- *"How long do I have to return these boots?"*
- *"What does the warranty cover?"*
- *"Where is my order?"*

Every one of these has an answer that already exists — in a policy document or in the
order database. A **copilot** can handle the repetitive questions instantly and
accurately, freeing the human team for the tricky cases.

That's what we're building. But first we need to understand the engine at the center of
it: the language model.

## 2. What an LLM is — and what it is *not*

An **LLM** (large language model) like Claude was trained on an enormous amount of public
text. That training gives it a remarkable ability to **understand and produce language**:
it can summarize, explain, rewrite, classify, and answer general-knowledge questions.

But two limits matter enormously for business:

- 🌐 **It only knows public information, up to a training cutoff.** It learned from the
  open web — not from *your* company's private documents or databases.
- 🕰️ **It doesn't know anything that happened after training, or anything specific to you.**
  Trailhead's return policy, today's inventory, the status of order #1007 — the model has
  simply never seen these.

So Claude is a brilliant *language* engine with **no knowledge of your business**. Keep
that picture in mind — everything we build this workshop is about closing that gap.

## 3. Set up your environment

We need two things: the Anthropic library (to talk to Claude) and your **API key**.

> 🔑 **Get an API key** at <https://console.anthropic.com/> → *API Keys*. Treat it like a
> password — never share it or paste it into a public place.

Run the cell below to install the library. (The `%pip` command works in both Google Colab
and Databricks. The `-q` just keeps the output quiet.)

In [ ]:
%pip install -q anthropic

Now enter your API key. Running this cell pops up a secure box — what you type **is not
saved in the notebook**, so it's safe to share your notebook afterward.

In [ ]:
import os, getpass

# getpass hides your key as you type, and we store it only for this session.
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Paste your Anthropic API key: ")

print("API key is set. ✅")

### Configuration

We keep the settings you might want to change in **one place**. Throughout the workshop,
look for a configuration cell like this near the top of each notebook.

- `MODEL` — which Claude model to use. We default to **Haiku 4.5**, the fastest and
  cheapest model. It's more than capable for this workshop, and keeps your total spend to
  a few cents. (You could switch to `"claude-sonnet-5"` or `"claude-opus-4-8"` for harder
  reasoning at higher cost — model choice is a real design decision you'll weigh in your
  own projects.)
- `MAX_TOKENS` — a cap on how long each reply can be. Smaller = cheaper and faster.

In [ ]:
MODEL = "claude-haiku-4-5"   # cheapest Claude model; great for this workshop
MAX_TOKENS = 300             # cap reply length (and cost)

## 4. Your first call to Claude

Talking to Claude is a single function call. You send a **message**, and you get back a
**response**. Let's ask something from general knowledge — the kind of thing an LLM *can*
answer on its own.

In [ ]:
import anthropic

# Create the client once. It automatically reads your ANTHROPIC_API_KEY.
client = anthropic.Anthropic()

response = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=[
        {"role": "user", "content": "In one sentence, what makes a good backpacking tent?"}
    ],
)

# The reply comes back as a list of content blocks; we want the text.
print(response.content[0].text)

🎉 That's a real AI response. A few things to notice:

- We passed a list of **messages**. Here there's just one, from the `"user"`.
- The reply lives in `response.content`. We pulled out `.text` from the first block.
- Every call also reports token **usage** — how you'd track cost. Let's peek:

In [ ]:
print("Input tokens: ", response.usage.input_tokens)
print("Output tokens:", response.usage.output_tokens)
# Haiku 4.5 pricing is about $1 per million input tokens and $5 per million output tokens,
# so a call like this costs a tiny fraction of a cent.

### A small helper to keep things tidy

We'll be calling Claude a lot. Let's wrap that same call in a short helper named `ask()` so
we don't repeat ourselves. (In later notebooks this exact helper lives in a shared file
called `trailhead.py` — same idea, just packaged for reuse.)

In [ ]:
def ask(prompt, system=None, max_tokens=MAX_TOKENS):
    """Send one prompt to Claude and return the reply text."""
    kwargs = {
        "model": MODEL,
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": prompt}],
    }
    if system:                       # an optional instruction that sets Claude's role
        kwargs["system"] = system
    response = client.messages.create(**kwargs)
    return "".join(block.text for block in response.content if block.type == "text")

# Quick test:
print(ask("Say hello to a group of workshop learners in one short sentence."))

## 5. Steering Claude with a *system prompt*

The **system prompt** is a separate instruction that sets Claude's role, tone, and rules —
before it ever sees the user's question. It's how you turn a general assistant into *your*
support agent.

Compare the two calls below: same question, different system prompt.

In [ ]:
question = "A customer asks: do you sell anything for cold-weather camping?"

print("--- No system prompt ---")
print(ask(question))

print("\n--- With a support-agent system prompt ---")
support_persona = (
    "You are a friendly customer support agent for Trailhead Supply Co., "
    "an online outdoor gear retailer. Keep answers short and helpful."
)
print(ask(question, system=support_persona))

> ✏️ **Try it.** Change `support_persona` above — make the agent extremely enthusiastic,
> or tell it to always end with a question. Re-run the cell and watch the tone shift. The
> system prompt is one of your most powerful (and cheapest) controls.

## 6. The catch: Claude doesn't know *your* business

The persona made Claude *sound* like a Trailhead agent. But sounding right and *being*
right are different things. Let's ask a question whose answer lives only in Trailhead's
**own policy documents** — something the model was never trained on.

In [ ]:
policy_question = (
    "For Trailhead Supply Co., can I return a Final Sale item, "
    "and if so how many days do I have?"
)

print(ask(policy_question, system=support_persona))

Look closely at that answer. You'll most likely see one of two things:

1. Claude **admits it doesn't have the information**, or
2. Claude gives a **confident, reasonable-sounding guess** (like "30 days") — which may be
   **completely wrong**.

Either way, this is a problem. The model never saw Trailhead's policy, so it can only guess.
And *a confident wrong answer about your policies is worse than no answer at all.*

> 🔑 **This is the central lesson of the whole workshop.** An LLM is a language expert with
> **no knowledge of your business**. To be useful, it needs to be given your facts.

## 7. Two ways to give a model your knowledge

How do we get Trailhead's real policies into Claude's answers? There are two broad options:

**Option A — Train (or fine-tune) a model.** Bake the knowledge into the model's internal
weights by training on your data.
- ❌ Expensive, slow, and requires machine-learning expertise.
- ❌ Goes stale the moment your data changes (new prices, new policy) — you'd have to
  retrain.
- ❌ Overkill for "answer questions from our documents."

**Option B — Ground the model at question time.** Leave the model exactly as it is, and
simply **include the relevant information in the prompt** when you ask. The model reads
your facts right there and answers from them.
- ✅ No training, no ML expertise, no waiting.
- ✅ Always current — you feed it today's data.
- ✅ This is how most real business copilots are built.

> 🧭 **We are not training a model.** We're **grounding** one — giving Claude the right
> information at the moment it answers. The rest of this workshop is about doing that
> *automatically* and *at scale*.

Let's prove Option B works. We'll paste the relevant slice of Trailhead's return policy
into the prompt, then ask the exact same question.

In [ ]:
# A snippet straight from Trailhead's return policy document.
policy_snippet = """
Final Sale & Clearance Items: Items marked Final Sale (typically clearance and heavily
discounted products) are not returnable or refundable. A product's Final Sale status is
shown on its product page and in your order confirmation.
"""

grounded_prompt = f"""Use ONLY the policy information below to answer the customer's question.
If the answer isn't in the policy, say you're not sure.

POLICY:
{policy_snippet}

CUSTOMER QUESTION:
Can I return a Final Sale item, and if so how many days do I have?
"""

print(ask(grounded_prompt, system=support_persona))

That's the difference. **Same model, no training** — we just handed Claude the right facts,
and now it answers correctly and specifically: Final Sale items can't be returned.

Notice two things we did in the prompt, which we'll reuse all workshop:
- We told Claude to **use only the provided information**, and
- We told it to **say "I'm not sure"** when the answer isn't there (a guardrail against
  guessing).

## 8. From "paste a snippet" to a real copilot

Pasting the right snippet by hand works — but it doesn't scale. Trailhead has many
documents and thousands of orders. The customer just asks a question; *something* has to
**find the right information automatically** and hand it to Claude.

That "something" is the copilot we'll build over the next five notebooks:

```
                    ┌─────────────────────────────────────────────┐
   Customer         │            TRAILHEAD COPILOT                 │
   question  ─────▶ │                                             │
                    │   ┌─────────┐   is this about a policy       │
                    │   │ ROUTER  │   or an order?  (Notebook 5)   │
                    │   └────┬────┘                                │
                    │        │                                     │
                    │   ┌────┴───────────────┐                     │
                    │   ▼                    ▼                     │
                    │  Policy question     Order / data question   │
                    │  ┌───────────────┐   ┌────────────────────┐  │
                    │  │ Search the     │   │ Look up the record │  │
                    │  │ documents (RAG)│   │ in the database    │  │
                    │  │ (Notebooks 2-3)│   │ (Notebook 4)       │  │
                    │  └───────┬───────┘   └─────────┬──────────┘  │
                    │          └─────────┬───────────┘             │
                    │                    ▼                         │
                    │            Give Claude the facts,            │
                    │            get a grounded answer             │
                    │            (+ evaluate & harden, Notebook 6) │
                    └─────────────────────┬───────────────────────┘
                                          ▼
                                  Accurate, grounded reply
```

Every box is just a smarter version of what you did by hand today: **get the right
information, then let Claude answer from it.** That pattern — retrieve, then generate — is
called **RAG (Retrieval-Augmented Generation)**, and it's the backbone of practical
business AI.

## 9. Recap

- An **LLM** is a powerful language engine trained on public data — it does **not** know
  your private or current business information.
- You talk to Claude by sending **messages** and reading the **response**; a **system
  prompt** sets its role and rules.
- Asked about Trailhead's own policy with no data, Claude **guessed** — a real risk.
- We fixed it by **grounding**: putting the relevant facts into the prompt. **No training
  required** — same model, new knowledge.
- Doing this automatically and at scale is **RAG**, the architecture we build next.

**Next up — Notebook 2: From Business Documents to Searchable Knowledge.** We'll teach the
copilot to *find* the right policy snippet on its own, instead of us pasting it.

## 10. Exercises

**Guided**
1. Re-run the ungrounded policy question (Section 6) two or three times. Does Claude answer
   the same way each time? What does that tell you about relying on it for facts?
2. In Section 7's grounded prompt, delete the line that says *"If the answer isn't in the
   policy, say you're not sure,"* then ask something the snippet doesn't cover (e.g.,
   *"how much is expedited shipping?"*). Compare the behavior.

**Challenge**
3. Write a grounded prompt that answers a **shipping** question. Copy a sentence or two of a
   made-up shipping rule into a `shipping_snippet`, paste it into the prompt like we did,
   and ask a matching question. You've just hand-built a mini-RAG answer.
4. Change `MODEL` to `"claude-sonnet-5"` and re-run one of the grounded prompts. Do you
   notice a difference in the answer? Was it worth the higher cost for this task?

## 11. Learn more

- **Anthropic — Intro to Claude & the Messages API:** <https://docs.claude.com/en/docs/intro-to-claude>
- **Anthropic — Prompt engineering overview:** <https://docs.claude.com/en/docs/build-with-claude/prompt-engineering/overview>
- **Anthropic — System prompts:** <https://docs.claude.com/en/docs/build-with-claude/prompt-engineering/system-prompts>
- **Codecademy — Intro to Generative AI:** <https://www.codecademy.com/learn/intro-to-generative-ai>
- **Codecademy — Learn Python 3** (if you'd like more Python footing): <https://www.codecademy.com/learn/learn-python-3>

---
> ### 👩‍🏫 Instructor notes (remove before distributing to students)
>
> - **Timing (~55 min):** ~10 min framing (§1–2), ~15 min setup + first call (§3–4),
>   ~15 min system prompts + the ungrounded/grounded contrast (§5–7), ~10 min architecture
>   + recap (§8–9), remainder for exercises.
> - **The money moment is §6→§7.** Run the ungrounded policy question live; whatever it
>   does (refuse *or* confidently guess) is a teaching win. Then reveal the grounded answer.
>   Emphasize repeatedly: *same model, no training — we just supplied the facts.*
> - If a learner's ungrounded answer happens to be correct-ish, re-run it a couple of times
>   or switch the question (e.g., "how many days to return worn boots?") to surface a guess.
> - **Cost:** the whole notebook is ~6 short Haiku calls — well under a cent per student.
> - Common setup snag: a mistyped/expired API key raises an `AuthenticationError`. Have
>   students re-run the key cell.
> - No repo clone is needed for this notebook — it only uses `anthropic` and inline
>   snippets. Notebook 2 introduces loading the Trailhead files.